### **Step 1: Set up Java in Google Colab**
Run the following commands

In [ ]:
!apt update -q
!apt-get install -q openjdk-11-jdk-headless


Then confirm Java is intalled

In [ ]:
!java -version

In [ ]:
!javac -version

### **Step 2: Create a Twelve Data Account**

Create a free Twelve Data account using this [link](https://twelvedata.com)

Generate an API key. you’ll use this key to retrieve market data.



### **Step 3: Store your API key securely**
Run this Python code, and enter your Secure API key when prompted

In [ ]:
from getpass import getpass
import os

os.environ["TWELVE_DATA_API_KEY"] = getpass("Enter your Twelve Data API key: ")

### **Step 4: Create the `App.java` file**
Run the cell to create an App.java file and write the code into the

In [ ]:
%%writefile App.java
import java.io.BufferedReader;
import java.io.IOException;
import java.io.InputStreamReader;
import java.net.HttpURLConnection;
import java.net.URL;
import java.time.LocalDateTime;
import java.util.LinkedList;

import java.util.Queue;

public class App {
    //getting the API Key
    private static final String API_KEY = System.getenv("TWELVE_DATA_API_KEY");
    private static final String GET_URL = "https://api.twelvedata.com/price?symbol=DIA&apikey="+API_KEY;
    //or i could hard code the api key into the URL variable as so (delete before submitting):
    //private static final String GET_URL = "https://api.twelvedata.com/price?symbol=DIA&apikey={pastemyurlhere}";
    private static final Queue<Double> resultQueue = new LinkedList<>();


    public static void main(String[] args) throws IOException {
        while(true){
            sendGetRequest();
            try {
                Thread.sleep((15000));
            } catch (InterruptedException e) {
                throw new RuntimeException(e);
            }
        }
    }

    private static void sendGetRequest() throws IOException {
        URL obj = new URL(GET_URL);
        HttpURLConnection connection = (HttpURLConnection) obj.openConnection();
        connection.setRequestMethod("GET");
        int responseCode = connection.getResponseCode();
        System.out.println("GET Response code: " + responseCode);
        //if the response code is successful
        if (responseCode == HttpURLConnection.HTTP_OK) { // success
            BufferedReader in = new BufferedReader(new InputStreamReader(connection.getInputStream()));
            String inputLine;
            StringBuffer response = new StringBuffer();

            while ((inputLine = in.readLine()) != null) {
                response.append(inputLine);
            }
            in.close();

            // parsing, storing and printing result
            String output =response.toString();
            Double price = getPriceFromOutput(output);
            resultQueue.add(price);
            System.out.println(resultQueue);

            LocalDateTime dateTimeStamp = LocalDateTime.now();
            System.out.println("Added data point: price="+ price +
                    ", timestamp=" + dateTimeStamp);
            System.out.println("Current queue size: " + resultQueue.size());
            System.out.println("Waiting 15 seconds to obtain the next value");


        } else {
            System.out.println("GET request did not work.");
        }




    }
    private static Double getPriceFromOutput(String JSONOutput){
        String marker = "\"price\":\"";

        int start = JSONOutput.indexOf(marker);
        start += marker.length();

        int end = JSONOutput.indexOf("\"", start);

        String price = JSONOutput.substring(start, end);

        return Double.parseDouble(price);
    }
}

### **Step 5: Compile and Run the `App.java` file**

Run the following two cells

In [ ]:
!javac App.java

In [ ]:
!java App

### **Step 6: Create your visualisation in Python**

Once the code is run, your output should include lines like:



```
Added data point: price=506.049988, timestamp=2026-05-26T10:28:36.547871559Z
```

Copy some of the lines that look like that, and assign it to the `java_output` variable (inside the triple quotes) in the cell below

After that is done, run the cell





In [ ]:
%%writefile GenerateGraph.py
import re
import pandas as pd
import matplotlib.pyplot as plt

# Paste your Java output between the triple quotes below
# This is an example output. Delete this and add your own results
java_output = """
Added data point: price=514.059998, timestamp=2026-06-02T22:15:34.163725528
Added data point: price=514.059998, timestamp=2026-06-02T22:15:49.351220437
Added data point: price=514.059998, timestamp=2026-06-02T22:16:04.580829727
Added data point: price=514.059998, timestamp=2026-06-02T22:16:19.758465166
Added data point: price=514.059998, timestamp=2026-06-02T22:16:34.972787298
"""

# Extract price and timestamp
matches = re.findall(
    r"price=([0-9.]+), timestamp=([^\n]+)",
    java_output
)

# Convert to DataFrame
df = pd.DataFrame(matches, columns=["price", "timestamp"])

# Convert data types
df["price"] = df["price"].astype(float)
df["timestamp"] = pd.to_datetime(df["timestamp"])

print(df)

# Plot the data
plt.figure(figsize=(10, 5))
plt.plot(df["timestamp"], df["price"], marker="o")

plt.title("DIA Price Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Price")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### **Step 4: Test your visualization**
Your graph should:



*   Display multiple data points
* Convert them into a structured format
* Show how the price changes over time

